# **Modelo Preditivo Imobiliário**

### Precificação de imóveis residenciais em King County

---

<table width="100%" cellpadding="12">
  <tr><th width="27%" align="left"><big>Identificação</big></th><th align="left"><big>Informação</big></th></tr>
  <tr><td><big><b>Curso</b></big></td><td><big>Desenvolvimento em IA para Análise Preditiva</big></td></tr>
  <tr><td><big><b>Projeto</b></big></td><td><big>Projeto Avaliativo Final — Módulo I</big></td></tr>
  <tr><td><big><b>Professor</b></big></td><td><big>Maurício Costa Cordeiro</big></td></tr>
  <tr><td><big><b>Aluno</b></big></td><td><big>Ricardo Pasquali dos Reis</big></td></tr>
  <tr><td><big><b>Dataset</b></big></td><td><big>King County House Sales — Opção A</big></td></tr>
  <tr><td><big><b>Ambiente</b></big></td><td><big>Google Colab</big></td></tr>
  <tr><td><big><b>Problema</b></big></td><td><big>Regressão supervisionada</big></td></tr>
  <tr><td><big><b>Variável-alvo</b></big></td><td><big><code>price</code>, preço de venda em dólares</big></td></tr>
</table>

---

## **Problema preditivo declarado**

Este projeto busca **prever a variável-alvo `price`**, um valor numérico contínuo que representa o preço de venda do imóvel em dólares, a partir de características físicas, temporais e de localização conhecidas antes da transação.

Isso importa para o negócio porque uma estimativa inicial consistente permite **priorizar imóveis, selecionar comparáveis e identificar preços possivelmente superavaliados ou subavaliados**. Uma diferença de dezenas de milhares de dólares afeta diretamente negociação, entrada, financiamento e exposição ao risco; por isso, o resultado será apresentado com métricas de erro e limites de uso, e não como um preço final automático.

> **Objetivo operacional:** produzir uma referência inicial de preço para apoiar a triagem imobiliária. A previsão não substitui vistoria, análise de comparáveis ou laudo, especialmente nos imóveis de maior valor.


# **1. Preparação do ambiente acadêmico**

A preparação limita-se às importações, às duas dependências geoespaciais, à definição dos caminhos e ao download validado da base pública. As funções analíticas são apresentadas somente nas fases em que passam a ser utilizadas.


In [ ]:
import hashlib, importlib.util, json, os, platform, shutil, subprocess, sys, tempfile, urllib.request
from pathlib import Path

faltantes = [pacote for modulo, pacote in {"h3": "h3>=4.1", "plotly": "plotly>=5.18", "jinja2": "jinja2>=3.1"}.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

import h3, joblib, matplotlib, matplotlib.pyplot as plt, numpy as np, pandas as pd, plotly
import scipy, seaborn as sns, sklearn
from IPython.display import FileLink, display
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

SEED, TEST_SIZE, CV_FOLDS, N_JOBS, TARGET = 42, 0.20, 5, 1, "price"
em_colab = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path(os.environ["PROJETO_ROOT"]) if os.environ.get("PROJETO_ROOT") else (Path("/content/modelo_preditivo_imobiliario") if em_colab else Path(tempfile.gettempdir()) / "modelo_preditivo_imobiliario")
RAW_DATA = ROOT / "data/raw/kc_house_data.csv"
PROCESSED_DATA, FINAL_DATA = ROOT / "data/processed/kc_house_processed.csv", ROOT / "data/final/kc_house_final.csv"
FIGURES_DIR, MAPS_DIR, MODEL_DIR = ROOT / "outputs/figures", ROOT / "outputs/maps", ROOT / "models/v1"
for pasta in [RAW_DATA.parent, PROCESSED_DATA.parent, FINAL_DATA.parent, FIGURES_DIR, MAPS_DIR, MODEL_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

def baixar_base():
    assinatura = "aa501a2b18e0d0a288a0fdeb043b2c89d6c7e2ead1f9588165537d4e4e12b9df"
    if RAW_DATA.exists() and hashlib.sha256(RAW_DATA.read_bytes()).hexdigest() == assinatura:
        return
    erros = []
    for url in [
        "https://drive.usercontent.google.com/download?id=1XMw1aKYqfFA13AbOzd9zRElv216bCxqZ&export=download&confirm=t",
        "https://drive.google.com/uc?export=download&id=1XMw1aKYqfFA13AbOzd9zRElv216bCxqZ",
    ]:
        parcial = RAW_DATA.with_suffix(".part")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as origem, parcial.open("wb") as destino:
                shutil.copyfileobj(origem, destino, 1024 * 1024)
            obtida = hashlib.sha256(parcial.read_bytes()).hexdigest()
            if obtida != assinatura:
                raise RuntimeError(f"SHA-256 inesperado: {obtida}")
            parcial.replace(RAW_DATA)
            return
        except Exception as erro:
            parcial.unlink(missing_ok=True)
            erros.append(f"{type(erro).__name__}: {erro}")
    raise RuntimeError("Falha no download da base pública: " + " | ".join(erros))

baixar_base()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
pd.set_option("display.max_columns", 30)
{"Python": platform.python_version(), "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "H3": h3.__version__, "Plotly": plotly.__version__}


> **Reprodutibilidade:** o notebook não depende de ZIP, repositório remoto ou módulos auxiliares. O CSV público é conferido por SHA-256; as funções permanecem visíveis junto às respectivas etapas e as versões do ambiente são registradas ao final.


# **2. Fase 1 — Análise Exploratória de Dados (EDA)**

A leitura inicial verifica dimensão, tipos, ausências, duplicidades, distribuição do preço e representatividade temporal antes de qualquer transformação.


In [ ]:
def carregar_dados(caminho=RAW_DATA) -> pd.DataFrame:
    return pd.read_csv(caminho)

def salvar_csv(df: pd.DataFrame, caminho) -> None:
    caminho.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(caminho, index=False)

def _quartos_incompativeis_com_area(df: pd.DataFrame) -> pd.Series:
    area_por_quarto = df["sqft_living"].div(df["bedrooms"].replace(0, np.nan))
    return df["bedrooms"].gt(10) & area_por_quarto.lt(100)

def auditar_dados(df: pd.DataFrame) -> dict:
    contagem_id = df.groupby("id").size()
    ano_venda = pd.to_numeric(df["date"].astype(str).str[:4], errors="coerce")
    return {
        "linhas": int(len(df)),
        "colunas": int(df.shape[1]),
        "duplicatas_exatas": int(df.duplicated().sum()),
        "ids_com_revenda": int(contagem_id.gt(1).sum()),
        "registros_adicionais_de_revenda": int(df.duplicated("id").sum()),
        "ausentes": {k: int(v) for k, v in df.isna().sum().items() if v},
        "quartos_incompativeis_com_area": int(_quartos_incompativeis_com_area(df).sum()),
        "vendas_antes_do_ano_de_construcao": int(df["yr_built"].gt(ano_venda).sum()),
    }

df_raw = carregar_dados()
auditoria = auditar_dados(df_raw)
display(pd.Series(auditoria, name="resultado").to_frame())
display(df_raw.head())
display(df_raw.dtypes.rename("tipo").to_frame())
display(df_raw.describe().T.style.format("{:,.2f}"))


A base reúne **21.613 vendas e 21 colunas**. O preço mediano é **US\$ 450 mil**, abaixo da média de aproximadamente **US\$ 540 mil**, primeira evidência da cauda à direita. Não há duplicatas exatas; IDs repetidos representam revendas e, por isso, serão mantidos e agrupados na separação dos dados.

## **2.1 Escopo e representatividade**

| Dimensão | Cobertura observada | Limite para uso |
|---|---|---|
| temporal | vendas registradas em 2014–2015 | não descreve inflação, juros ou preferências atuais |
| geográfica | King County, Washington | não deve ser transferido automaticamente para outros mercados |
| população | imóveis efetivamente vendidos | não representa anúncios retirados, imóveis sem venda ou todo o estoque residencial |
| finalidade | referência inicial de preço | não substitui vistoria, comparáveis recentes ou laudo |

O projeto é, portanto, um **benchmark histórico e acadêmico**. A validação mede generalização dentro dessa população observada; ela não comprova validade temporal para o mercado atual.


## **2.2 Visualizações e leitura espacial**

As saídas foram separadas em três blocos para que cada evidência tenha sua própria área de visualização no Colab. O primeiro compara distribuição e relações com o preço; o segundo reúne a matriz e as correlações; o terceiro concentra o mapa e o painel geoespacial.


In [ ]:
#@title Rotinas dos gráficos e correlações — apoio
from pathlib import Path
CORES = {"azul": "#1d4ed8", "laranja": "#ea580c", "verde": "#15803d"}
def _salvar(fig, nome: str, pasta: Path = FIGURES_DIR):
    pasta.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(pasta / nome, dpi=160, bbox_inches="tight")
    return fig

def plot_distribuicao_preco(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    sns.histplot(df["price"], bins=55, ax=axes[0], color=CORES["azul"])
    axes[0].set(title="Distribuição do preço", xlabel="Preço (US$)", ylabel="Imóveis")
    sns.histplot(np.log1p(df["price"]), bins=55, ax=axes[1], color=CORES["verde"])
    axes[1].set(title="Distribuição após log1p", xlabel="log1p(preço)", ylabel="Imóveis")
    return _salvar(fig, "01_distribuicao_preco.png")

def plot_relacoes_preco(df: pd.DataFrame):
    amostra = df.sample(min(5_000, len(df)), random_state=42)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    sns.scatterplot(data=amostra, x="sqft_living", y="price", alpha=0.35, ax=axes[0])
    axes[0].set(title="Área construída × preço", xlabel="Área construída (ft²)", ylabel="Preço (US$)")
    sns.boxplot(data=df, x="grade", y="price", color="#93c5fd", showfliers=False, ax=axes[1])
    axes[1].set(title="Padrão construtivo × preço", xlabel="Grade", ylabel="Preço (US$)")
    return _salvar(fig, "02_relacoes_preco.png")

def plot_correlacao(df: pd.DataFrame):
    colunas = [
        "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
        "waterfront", "view", "condition", "grade", "sqft_above", "sqft_basement",
        "yr_built", "lat", "long", "sqft_living15", "sqft_lot15",
    ]
    fig, ax = plt.subplots(figsize=(12.5, 9.5))
    sns.heatmap(df[colunas].corr(), cmap="vlag", center=0, square=True, ax=ax)
    ax.set_title("Correlação de Pearson entre variáveis numéricas")
    return _salvar(fig, "03_correlacao_pearson.png")

def comparar_correlacoes(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    base = df[colunas + [TARGET]]
    resultado = pd.DataFrame(
        {
            "Pearson": base.corr(method="pearson")[TARGET].drop(TARGET),
            "Spearman": base.corr(method="spearman")[TARGET].drop(TARGET),
        }
    )
    resultado["diferenca_absoluta"] = resultado["Spearman"].abs() - resultado["Pearson"].abs()
    return resultado.sort_values("Spearman", key=lambda s: s.abs(), ascending=False)

def resumir_distribuicao_alvo(serie: pd.Series) -> pd.DataFrame:
    """Quantifica a cauda do preço antes e depois da transformação logarítmica."""
    escalas = {"price": serie.astype(float), "log1p(price)": np.log1p(serie)}
    return pd.DataFrame([
        {
            "escala": nome,
            "media": valores.mean(),
            "mediana": valores.median(),
            "assimetria": valores.skew(),
            "curtose_excesso": valores.kurt(),
        }
        for nome, valores in escalas.items()
    ])


### **2.2.1 Distribuição e relações com o preço**

O preço é observado na escala original e após `log1p`. A **assimetria** mede o desequilíbrio das caudas: valor próximo de zero indica maior simetria e valores positivos elevados indicam cauda longa à direita. A **curtose excedente** usa zero como referência; valores muito altos sinalizam concentração central combinada com observações extremas.

Essas medidas não funcionam como teste de normalidade. Aqui, elas servem para verificar se a transformação logarítmica realmente reduz a influência desproporcional dos imóveis mais caros antes da comparação dos modelos.


In [ ]:
diagnostico_alvo = resumir_distribuicao_alvo(df_raw[TARGET])
diagnostico_alvo.to_csv(ROOT / "outputs/diagnostico_distribuicao_alvo.csv", index=False)
display(diagnostico_alvo.style.format({"media": "{:,.2f}", "mediana": "{:,.2f}", "assimetria": "{:.3f}", "curtose_excesso": "{:.3f}"}))
plot_distribuicao_preco(df_raw); plt.show()
plot_relacoes_preco(df_raw); plt.show()


Na escala original, a assimetria é **4,024** e a curtose excedente chega a **34,586**. Após `log1p`, elas caem para **0,428** e **0,692**. A redução quantifica o que o histograma sugere: o log aproxima a massa central e impede que poucas transações milionárias dominem o ajuste, sem alterar a ordem relativa dos imóveis.

A área construída apresenta associação crescente com o valor, enquanto `grade` separa faixas de padrão construtivo. Esses sinais justificam confrontar um palpite ingênuo, um baseline linear e modelos capazes de representar curvaturas e interações.


### **2.2.2 Matriz e correlações Pearson × Spearman**

Pearson resume associação linear; Spearman mede associação monotônica pelos postos e é menos dependente da forma linear. Correlação descreve associação entre variáveis, enquanto regressão estima o alvo usando várias variáveis em conjunto — distinção enfatizada na Semana 11 do curso.


In [ ]:
CORR_COLS = ["sqft_living", "grade", "sqft_living15", "bathrooms", "view", "bedrooms", "lat", "waterfront", "idade_imovel", "log_sqft_lot"]
plot_correlacao(df_raw); plt.show()

ano_venda = pd.to_numeric(df_raw["date"].astype(str).str[:4], errors="coerce")
df_corr = df_raw.assign(
    idade_imovel=(ano_venda - df_raw["yr_built"]).clip(lower=0),
    log_sqft_lot=np.log1p(df_raw["sqft_lot"]),
)
correlacoes = comparar_correlacoes(df_corr, CORR_COLS)
correlacoes.to_csv(ROOT / "outputs/correlacoes_pearson_spearman.csv")
display(correlacoes.style.format("{:.4f}"))


**`sqft_living` e `grade` concentram as associações mais fortes com o preço.** Em `lat`, Spearman supera Pearson, sinal de uma relação espacial monotônica que uma reta isolada não representa adequadamente. A matriz também expõe redundância entre medidas de área; por isso, a etapa seguinte usa VIF antes da modelagem.


### **2.2.3 Mapa e painel geoespacial**

O valor por metro quadrado é agregado por células H3 com mínimo de 20 vendas. A mediana reduz a influência de negócios extremos, e o contorno dourado identifica o decil regional superior. Essa variável é usada somente para inteligência territorial: como deriva de `price`, não entra no modelo preditivo.


In [ ]:
#@title Rotinas do painel geoespacial — apoio
import plotly.express as px
import plotly.graph_objects as go

H3_RESOLUTION, H3_MIN_SALES = 7, 20
PANEL_PATH = MAPS_DIR / 'painel_geoespacial.html'
ATLANTIC = ['#eaf1f4','#c7dce1','#91bbc1','#5e959f','#356c7b','#21485f','#132f48','#0b2036']
PANEL_TEMPLATE = '<!doctype html>\n<html lang="pt-BR">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width,initial-scale=1">\n  <title>Painel geoespacial — King County</title>\n  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">\n  <style>\n    :root{--ink:#172033;--muted:#667085;--line:#d8e1ea;--paper:#fff;--canvas:#eef3f7;--soft:#f7fafc;--petrol:#21485f;--deep:#0b2036;--gold:#b08d57}\n    *{box-sizing:border-box}body{margin:0;background:linear-gradient(145deg,#edf3f7,#f7f9fb 54%,#e9f0f3);color:var(--ink);font:15px/1.5 Inter,system-ui,"Segoe UI",sans-serif}\n    main{max-width:1460px;margin:22px auto;padding:26px;background:var(--paper);border:1px solid var(--line);border-radius:20px;box-shadow:0 18px 52px #0b203619}\n    h1{margin:0;font-size:clamp(1.85rem,4vw,2.65rem);letter-spacing:-.04em}.lead{margin:5px 0 18px;color:var(--muted)}\n    .toolbar{display:grid;grid-template-columns:1.25fr 1.2fr 1fr 1fr;gap:10px;padding:14px;background:var(--soft);border:1px solid var(--line);border-radius:13px}\n    label{display:grid;gap:4px;font-size:.82rem;font-weight:650;color:#344054}.check{display:flex;align-items:center;gap:7px;min-height:38px}\n    select,input[type=search]{width:100%;min-height:38px;padding:7px 9px;border:1px solid #c7d2dc;border-radius:8px;background:#fff;color:var(--ink)}input[type=range]{width:100%}\n    button{min-height:38px;padding:7px 12px;border:1px solid #c7d2dc;border-radius:8px;background:#fff;color:var(--ink);cursor:pointer}button:hover{background:#eaf1f4}button:focus-visible,input:focus-visible,select:focus-visible{outline:3px solid #91bbc1;outline-offset:1px}button:disabled,select:disabled,input:disabled{opacity:.48;cursor:not-allowed}\n    .actions{display:flex;gap:8px;align-items:end;grid-column:span 2}.actions button{flex:1}.cards{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin:12px 0}.card{padding:12px 14px;border:1px solid var(--line);border-radius:11px;background:#fff}.card span{display:block;color:var(--muted);font-size:.82rem}.card b{font-size:1.38rem}\n    .layout{display:grid;grid-template-columns:minmax(0,2.25fr) minmax(280px,.75fr);gap:14px}.mapbox{position:relative}#map{height:690px;border:1px solid var(--line);border-radius:13px;background:#dce6ee;box-shadow:inset 0 0 0 1px #ffffff55}.map-fit{position:absolute;z-index:650;top:12px;right:12px;background:#fffffff2;border-color:#aebdca;box-shadow:0 7px 20px #0b203626;font-weight:700;backdrop-filter:blur(8px)}.legend{position:absolute;z-index:500;left:14px;bottom:66px;min-width:245px;max-width:310px;padding:10px 12px;background:#fffffff0;border:1px solid var(--line);border-radius:10px;box-shadow:0 8px 24px #0b203620;backdrop-filter:blur(8px)}.legend-block+.legend-block{margin-top:10px;padding-top:9px;border-top:1px solid var(--line)}.gradient{height:9px;margin:6px 0;border-radius:999px;background:linear-gradient(90deg,#eaf1f4,#c7dce1,#91bbc1,#5e959f,#356c7b,#21485f,#132f48,#0b2036)}.legend-values{display:flex;justify-content:space-between;gap:12px;font-size:.76rem;color:var(--muted)}.gold-key{display:flex;gap:7px;align-items:center;margin-top:7px;color:var(--muted);font-size:.76rem}.gold-key i{width:16px;height:9px;border:2px solid var(--gold);border-radius:2px}\n    .selected{min-height:46px;margin-top:8px;padding:10px 12px;border:1px solid var(--line);border-radius:10px;color:var(--muted);background:var(--soft)}\n    aside{border:1px solid var(--line);border-radius:13px;padding:15px;background:#fff}aside h2{margin:0 0 2px;font-size:1.2rem}aside .sub{margin:0 0 13px;color:var(--muted);font-size:.82rem}.rank{display:grid;grid-template-columns:22px minmax(0,1fr);gap:7px;align-items:center;margin:8px 0}.rank-num{font-weight:750;color:var(--gold);text-align:right}.rank button{width:100%;text-align:left;padding:8px 9px}.rank b{display:flex;justify-content:space-between;gap:8px}.rank small{display:block;color:var(--muted);margin-top:2px}.track{display:block;height:4px;margin-top:6px;background:#eaf0f3;border-radius:99px;overflow:hidden}.bar{display:block;height:100%;background:linear-gradient(90deg,#5e959f,#21485f)}\n    .note{margin-top:14px;padding:12px;background:#f5f1e9;border-left:4px solid var(--gold);border-radius:7px;font-size:.82rem;color:#59472b}.footer{margin:14px 2px 0;color:var(--muted);font-size:.78rem}\n    .leaflet-tooltip{border:1px solid var(--line);box-shadow:0 8px 24px #17203324;border-radius:8px;padding:9px 11px}.leaflet-control-attribution{font-size:10px}\n    @media(max-width:1080px){.toolbar{grid-template-columns:repeat(2,1fr)}.layout{grid-template-columns:1fr}aside{order:2}}\n    @media(max-width:650px){main{margin:0;padding:14px;border:0;border-radius:0}.toolbar{grid-template-columns:1fr}.cards{grid-template-columns:1fr}#map{height:590px}.legend{right:10px;left:10px;max-width:none}.actions{width:100%;grid-column:auto;flex-wrap:wrap}.actions button{flex:1 1 100%}}\n  </style>\n</head>\n<body><main>\n  <h1>Painel geoespacial de valor residencial</h1>\n  <p class="lead">King County · duas leituras complementares reconstruídas a partir das 21.613 vendas</p>\n\n  <section class="toolbar" aria-label="Filtros do painel">\n    <label>Camada\n      <select id="layer-mode">\n        <option value="h3">H3 — valor regional por m²</option>\n        <option value="transactions">Transações — preço absoluto</option>\n        <option value="both">Ambas as camadas</option>\n      </select>\n    </label>\n    <label>Métrica regional\n      <select id="metric">\n        <option value="valor_m2_mediano">Valor mediano (US$/m²)</option>\n        <option value="preco_mediano">Preço mediano (US$)</option>\n        <option value="vendas">Quantidade de vendas</option>\n        <option value="iqr_valor_m2">Dispersão IQR (US$/m²)</option>\n      </select>\n    </label>\n    <label>Mínimo de vendas H3: <span id="min-value">20</span>\n      <input id="min-sales" type="range" min="20" max="100" step="5" value="20">\n    </label>\n    <label>ZIP\n      <input id="zip" type="search" inputmode="numeric" placeholder="Todos ou 98004" maxlength="5">\n    </label>\n    <label>Mapa-base\n      <select id="basemap"><option value="light">Claro</option><option value="osm">OpenStreetMap</option><option value="dark">Escuro</option><option value="satellite">Satélite</option></select>\n    </label>\n    <label class="check"><input id="top-only" type="checkbox">Somente decil superior da camada</label>\n    <div class="actions"><button id="fit" type="button">Ajustar às vendas</button><button id="reset" type="button">Restaurar</button><button id="export" type="button">Exportar CSV</button></div>\n  </section>\n\n  <section class="cards" aria-live="polite">\n    <div class="card"><span id="stat-label-1">Células visíveis</span><b id="stat-1">290</b></div>\n    <div class="card"><span id="stat-label-2">Vendas representadas</span><b id="stat-2">20.014</b><span id="stat-detail-2">92,6% da base</span></div>\n    <div class="card"><span id="stat-label-3">Mediana das células</span><b id="stat-3">US$ 2.625/m²</b></div>\n  </section>\n\n  <section class="layout">\n    <div class="mapbox">\n      <div id="map" aria-label="Mapa navegável do valor residencial"></div>\n      <button id="fit-map" class="map-fit" type="button" aria-label="Ajustar o mapa à área com vendas visíveis">Ajustar mapa</button>\n      <div class="legend" aria-label="Legenda">\n        <div class="legend-block" id="legend-h3"><b id="legend-title-h3">Valor mediano (US$/m²)</b><div class="gradient"></div><div class="legend-values"><span id="legend-min-h3">US$ 1.572/m²</span><span id="legend-max-h3">US$ 4.297/m²</span></div><span class="gold-key"><i></i>Contorno: decil superior H3</span></div>\n        <div class="legend-block" id="legend-points"><b>Preço de venda · escala logarítmica</b><div class="gradient"></div><div class="legend-values"><span id="legend-min-points">US$ 75 mil</span><span id="legend-max-points">US$ 7,7 mi</span></div></div>\n      </div>\n      <div class="selected" id="selected">Clique em uma célula ou transação para fixar os detalhes.</div>\n    </div>\n    <aside><h2 id="ranking-title">Maiores valores no filtro</h2><p class="sub" id="ranking-subtitle">O ranking responde à métrica e à amostra visível.</p><div id="ranking"></div><div class="note" id="note">O contorno dourado identifica o decil superior do valor regional por m². Células abaixo da amostra mínima não são exibidas.</div></aside>\n  </section>\n  <p class="footer">Geometria territorial: US Census / us-atlas · valores: King County House Sales (2014–2015). Os mapas-base dependem de conexão à internet e mantêm a atribuição de seus provedores.</p>\n</main>\n<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>\n<script>\n(() => {\n  const regions = __REGIONS_JSON__;\n  const transactions = __TRANSACTIONS_JSON__;\n  const county = __COUNTY_JSON__;\n  const totalSales = transactions.length;\n  const palette = ["#eaf1f4","#c7dce1","#91bbc1","#5e959f","#356c7b","#21485f","#132f48","#0b2036"];\n  const gold = "#b08d57";\n  transactions.forEach((row,index) => { row.id = index; });\n  const fmt0 = (v) => new Intl.NumberFormat("pt-BR",{maximumFractionDigits:0}).format(v);\n  const fmt1 = (v) => new Intl.NumberFormat("pt-BR",{minimumFractionDigits:1,maximumFractionDigits:1}).format(v);\n  const moneyShort = (v) => v >= 1e6 ? `US$ ${fmt1(v/1e6)} mi` : `US$ ${fmt0(v/1e3)} mil`;\n  const metrics = {\n    valor_m2_mediano:{label:"Valor mediano",legend:"Valor mediano (US$/m²)",format:(v)=>`US$ ${fmt0(v)}/m²`},\n    preco_mediano:{label:"Preço mediano",legend:"Preço mediano (US$)",format:(v)=>`US$ ${fmt0(v)}`},\n    vendas:{label:"Quantidade de vendas",legend:"Quantidade de vendas",format:(v)=>`${fmt0(v)} vendas`},\n    iqr_valor_m2:{label:"Dispersão IQR",legend:"Dispersão IQR (US$/m²)",format:(v)=>`US$ ${fmt0(v)}/m²`}\n  };\n  const q = (values,p) => { const s=[...values].sort((a,b)=>a-b); if(!s.length)return 0; const x=(s.length-1)*p,i=Math.floor(x),r=x-i; return s[i+1]===undefined?s[i]:s[i]+r*(s[i+1]-s[i]); };\n  const priceMin = Math.min(...transactions.map((row)=>row.price));\n  const priceMax = Math.max(...transactions.map((row)=>row.price));\n  const priceTop = q(transactions.map((row)=>row.price),.90);\n  const els = {\n    mode:document.querySelector("#layer-mode"), metric:document.querySelector("#metric"), min:document.querySelector("#min-sales"), minValue:document.querySelector("#min-value"), zip:document.querySelector("#zip"), base:document.querySelector("#basemap"), top:document.querySelector("#top-only"), fit:document.querySelector("#fit"), fitMap:document.querySelector("#fit-map"), reset:document.querySelector("#reset"), export:document.querySelector("#export"),\n    statLabel1:document.querySelector("#stat-label-1"),stat1:document.querySelector("#stat-1"),statLabel2:document.querySelector("#stat-label-2"),stat2:document.querySelector("#stat-2"),statDetail2:document.querySelector("#stat-detail-2"),statLabel3:document.querySelector("#stat-label-3"),stat3:document.querySelector("#stat-3"),\n    legendH3:document.querySelector("#legend-h3"),legendPoints:document.querySelector("#legend-points"),legendTitleH3:document.querySelector("#legend-title-h3"),legendMinH3:document.querySelector("#legend-min-h3"),legendMaxH3:document.querySelector("#legend-max-h3"),legendMinPoints:document.querySelector("#legend-min-points"),legendMaxPoints:document.querySelector("#legend-max-points"),selected:document.querySelector("#selected"),ranking:document.querySelector("#ranking"),rankingTitle:document.querySelector("#ranking-title"),rankingSubtitle:document.querySelector("#ranking-subtitle"),note:document.querySelector("#note")\n  };\n\n  const map = L.map("map",{zoomControl:true,preferCanvas:true});\n  const bases = {\n    light:L.tileLayer("https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",{maxZoom:20,attribution:"&copy; OpenStreetMap &copy; CARTO"}),\n    osm:L.tileLayer("https://tile.openstreetmap.org/{z}/{x}/{y}.png",{maxZoom:19,attribution:"&copy; OpenStreetMap contributors"}),\n    dark:L.tileLayer("https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",{maxZoom:20,attribution:"&copy; OpenStreetMap &copy; CARTO"}),\n    satellite:L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",{maxZoom:19,attribution:"Tiles &copy; Esri"})\n  };\n  bases.light.addTo(map);\n  const countyLayer = L.geoJSON(county,{interactive:false,style:{color:"#607487",weight:1.2,fillColor:"#dbe5eb",fillOpacity:.10}}).addTo(map);\n  const countyBounds = countyLayer.getBounds();\n  map.fitBounds(countyBounds.pad(.025));\n  const renderer = L.canvas({padding:.5,tolerance:5});\n  const cells = L.layerGroup().addTo(map);\n  const points = L.layerGroup().addTo(map);\n  let cellsById = new Map(), pointsById = new Map(), currentBase = "light", selected = null;\n\n  const filteredRegions = () => { const min=Number(els.min.value),zip=els.zip.value.trim(); return regions.filter((row)=>row.vendas>=min&&(!zip||String(row.zipcode_dominante).includes(zip))&&(!els.top.checked||row.top_decil)); };\n  const filteredTransactions = () => { const zip=els.zip.value.trim(); return transactions.filter((row)=>(!zip||row.zipcode.includes(zip))&&(!els.top.checked||row.price>=priceTop)); };\n  const relevantBounds = () => {\n    const coordinates=[];\n    if(els.mode.value==="h3")filteredRegions().forEach((row)=>row.boundary.forEach(([lon,lat])=>coordinates.push([lat,lon])));\n    else filteredTransactions().forEach((row)=>coordinates.push([row.lat,row.lon]));\n    const bounds=L.latLngBounds(coordinates);\n    return bounds.isValid()?bounds:countyBounds;\n  };\n  const fitRelevant = () => { map.stop(); map.fitBounds(relevantBounds().pad(.06),{maxZoom:13,animate:true,duration:.45}); };\n  const linearColor = (value,low,high) => { const t=Math.max(0,Math.min(.999,(value-low)/(high-low||1))); return palette[Math.floor(t*palette.length)]; };\n  const logColor = (value) => { const t=Math.max(0,Math.min(.999,(Math.log(value)-Math.log(priceMin))/(Math.log(priceMax)-Math.log(priceMin)))); return palette[Math.floor(t*palette.length)]; };\n  const tooltipRegion = (row,metric) => `<b>ZIP ${row.zipcode_dominante}</b><br>${metrics[metric].label}: ${metrics[metric].format(row[metric])}<br>${fmt0(row.vendas)} vendas<br>Preço mediano: US$ ${fmt0(row.preco_mediano)}<br>Área mediana: ${fmt1(row.area_mediana_m2)} m²`;\n  const tooltipTransaction = (row) => `<b>Venda · ZIP ${row.zipcode}</b><br>Preço: US$ ${fmt0(row.price)}<br>Valor: US$ ${fmt0(row.valor_m2)}/m²<br>${row.date} · ${fmt0(row.sqft_living)} ft²<br>${row.bedrooms} quartos · ${fmt1(row.bathrooms)} banheiros`;\n\n  function selectRegion(row){ selected={kind:"h3",id:row.h3}; render(); const layer=cellsById.get(row.h3); if(layer){map.fitBounds(layer.getBounds().pad(2.2),{maxZoom:12});layer.openTooltip();} }\n  function selectTransaction(row){ selected={kind:"transaction",id:row.id}; render(); const layer=pointsById.get(row.id); if(layer){map.setView(layer.getLatLng(),14);layer.openTooltip();} }\n\n  function renderCells(rows,metric,low,high,combined){\n    cells.clearLayers(); cellsById=new Map();\n    rows.forEach((row)=>{\n      const geometry={type:"Feature",properties:{h3:row.h3},geometry:{type:"Polygon",coordinates:[row.boundary]}};\n      const isSelected=selected?.kind==="h3"&&selected.id===row.h3;\n      const layer=L.geoJSON(geometry,{style:{color:isSelected?"#2c6f8a":row.top_decil?gold:"#f8fafc",weight:isSelected?3:row.top_decil?1.8:.6,fillColor:linearColor(row[metric],low,high),fillOpacity:combined?.50:.76}});\n      layer.bindTooltip(tooltipRegion(row,metric),{sticky:true,direction:"top"});\n      layer.on("click",()=>selectRegion(row)); layer.addTo(cells); cellsById.set(row.h3,layer);\n    });\n  }\n\n  function renderPoints(rows,combined){\n    points.clearLayers(); pointsById=new Map();\n    rows.forEach((row)=>{\n      const isSelected=selected?.kind==="transaction"&&selected.id===row.id;\n      const marker=L.circleMarker([row.lat,row.lon],{renderer,radius:isSelected?5.5:2.4,color:isSelected?gold:logColor(row.price),weight:isSelected?2:0,fillColor:logColor(row.price),fillOpacity:combined?.62:.54});\n      marker.bindTooltip(tooltipTransaction(row),{sticky:true,direction:"top"});\n      marker.on("click",()=>selectTransaction(row)); marker.addTo(points); pointsById.set(row.id,marker);\n    });\n  }\n\n  function addRankRegion(row,index,max,metric){\n    const wrap=document.createElement("div"); wrap.className="rank"; wrap.innerHTML=`<span class="rank-num">${index+1}</span>`;\n    const button=document.createElement("button"); button.type="button"; button.innerHTML=`<b><span>ZIP ${row.zipcode_dominante}</span><span>${metrics[metric].format(row[metric])}</span></b><small>${fmt0(row.vendas)} vendas</small><span class="track"><span class="bar" style="width:${100*row[metric]/max}%"></span></span>`; button.addEventListener("click",()=>selectRegion(row)); wrap.appendChild(button); els.ranking.appendChild(wrap);\n  }\n  function addRankTransaction(row,index,max){\n    const wrap=document.createElement("div"); wrap.className="rank"; wrap.innerHTML=`<span class="rank-num">${index+1}</span>`;\n    const button=document.createElement("button"); button.type="button"; button.innerHTML=`<b><span>ZIP ${row.zipcode}</span><span>${moneyShort(row.price)}</span></b><small>${row.date} · US$ ${fmt0(row.valor_m2)}/m²</small><span class="track"><span class="bar" style="width:${100*row.price/max}%"></span></span>`; button.addEventListener("click",()=>selectTransaction(row)); wrap.appendChild(button); els.ranking.appendChild(wrap);\n  }\n\n  function render(){\n    const mode=els.mode.value,metric=els.metric.value,showH3=mode!=="transactions",showPoints=mode!=="h3",regionRows=filteredRegions(),transactionRows=filteredTransactions(),values=regionRows.map((row)=>Number(row[metric])),low=q(values,.05),high=q(values,.95);\n    els.metric.disabled=!showH3; els.min.disabled=!showH3; els.minValue.textContent=els.min.value;\n    if(showH3)renderCells(regionRows,metric,low,high,mode==="both");else{cells.clearLayers();cellsById=new Map();}\n    if(showPoints)renderPoints(transactionRows,mode==="both");else{points.clearLayers();pointsById=new Map();}\n    els.legendH3.hidden=!showH3; els.legendPoints.hidden=!showPoints; els.legendTitleH3.textContent=metrics[metric].legend; els.legendMinH3.textContent=regionRows.length?metrics[metric].format(low):"—"; els.legendMaxH3.textContent=regionRows.length?metrics[metric].format(high):"—"; els.legendMinPoints.textContent=moneyShort(priceMin);els.legendMaxPoints.textContent=moneyShort(priceMax);\n    els.ranking.replaceChildren();\n    if(mode==="transactions"){\n      els.statLabel1.textContent="Transações visíveis";els.stat1.textContent=fmt0(transactionRows.length);els.statLabel2.textContent="Preço mediano";els.stat2.textContent=transactionRows.length?moneyShort(q(transactionRows.map((row)=>row.price),.5)):"—";els.statDetail2.textContent=`${fmt1(100*transactionRows.length/totalSales)}% da base`;els.statLabel3.textContent="Valor mediano por m²";els.stat3.textContent=transactionRows.length?`US$ ${fmt0(q(transactionRows.map((row)=>row.valor_m2),.5))}/m²`:"—";\n      els.rankingTitle.textContent="Maiores vendas no filtro";els.rankingSubtitle.textContent="Ranking por preço absoluto da transação.";els.note.textContent="Cada ponto representa uma venda. A cor segue o preço absoluto em escala logarítmica, preservando simultaneamente imóveis usuais e vendas de alto padrão.";\n      const ranked=[...transactionRows].sort((a,b)=>b.price-a.price).slice(0,8),max=ranked[0]?.price||1;ranked.forEach((row,index)=>addRankTransaction(row,index,max));\n    }else{\n      const sales=regionRows.reduce((sum,row)=>sum+row.vendas,0);\n      els.statLabel1.textContent=mode==="both"?"Células H3 visíveis":"Células visíveis";els.stat1.textContent=fmt0(regionRows.length);els.statLabel2.textContent=mode==="both"?"Transações visíveis":"Vendas representadas";els.stat2.textContent=fmt0(mode==="both"?transactionRows.length:sales);els.statDetail2.textContent=`${fmt1(100*(mode==="both"?transactionRows.length:sales)/totalSales)}% da base`;els.statLabel3.textContent=`Mediana · ${metrics[metric].label}`;els.stat3.textContent=regionRows.length?metrics[metric].format(q(values,.5)):"—";\n      els.rankingTitle.textContent="Maiores células H3";els.rankingSubtitle.textContent=mode==="both"?"O ranking regional permanece legível sobre as vendas individuais.":"O ranking responde à métrica e à amostra visível.";els.note.textContent=mode==="both"?"A malha H3 resume o padrão regional por m²; os pontos exibem as vendas individuais. As duas cores usam a mesma linguagem visual, mas representam métricas diferentes.":"O contorno dourado identifica o decil superior do valor regional por m². Células abaixo da amostra mínima não são exibidas.";\n      const ranked=[...regionRows].sort((a,b)=>b[metric]-a[metric]).slice(0,8),max=ranked[0]?.[metric]||1;ranked.forEach((row,index)=>addRankRegion(row,index,max,metric));\n    }\n    if(!els.ranking.children.length)els.ranking.textContent="Nenhum registro atende aos filtros.";\n    const validSelection=selected&&(selected.kind==="h3"?regionRows.some((row)=>row.h3===selected.id):transactionRows.some((row)=>row.id===selected.id));\n    if(!validSelection){selected=null;els.selected.textContent="Clique em uma célula ou transação para fixar os detalhes.";}else if(selected.kind==="h3"){\n      const row=regionRows.find((item)=>item.h3===selected.id);els.selected.innerHTML=`<b>ZIP ${row.zipcode_dominante}</b> · célula H3 · ${metrics[metric].format(row[metric])} · ${fmt0(row.vendas)} vendas`;\n    }else{\n      const row=transactionRows.find((item)=>item.id===selected.id);els.selected.innerHTML=`<b>ZIP ${row.zipcode}</b> · venda de US$ ${fmt0(row.price)} · US$ ${fmt0(row.valor_m2)}/m² · ${row.date}`;\n    }\n  }\n\n  const csvValue=(value)=>{const text=String(value??"");return /[",\\n]/.test(text)?`"${text.replaceAll(\'"\',\'""\')}"`:text;};\n  const downloadCSV=(name,columns,rows)=>{const csv=[columns.join(","),...rows.map((row)=>columns.map((column)=>csvValue(row[column])).join(","))].join("\\n"),url=URL.createObjectURL(new Blob([csv],{type:"text/csv;charset=utf-8"})),link=document.createElement("a");link.href=url;link.download=name;link.click();URL.revokeObjectURL(url);};\n  [els.mode,els.metric,els.min,els.zip,els.top].forEach((control)=>control.addEventListener("input",render));\n  els.base.addEventListener("change",()=>{map.removeLayer(bases[currentBase]);currentBase=els.base.value;bases[currentBase].addTo(map);bases[currentBase].bringToBack();countyLayer.bringToFront();cells.eachLayer((layer)=>layer.bringToFront&&layer.bringToFront());points.eachLayer((layer)=>layer.bringToFront&&layer.bringToFront());});\n  els.fit.addEventListener("click",fitRelevant);\n  els.fitMap.addEventListener("click",fitRelevant);\n  els.reset.addEventListener("click",()=>{els.mode.value="h3";els.metric.value="valor_m2_mediano";els.min.value="20";els.zip.value="";els.top.checked=false;selected=null;map.fitBounds(countyBounds.pad(.025));render();});\n  els.export.addEventListener("click",()=>{\n    if(els.mode.value==="transactions")downloadCSV("transacoes_filtradas.csv",["date","zipcode","price","valor_m2","lat","lon","sqft_living","bedrooms","bathrooms"],filteredTransactions());\n    else downloadCSV("regioes_h3_filtradas.csv",["h3","zipcode_dominante","vendas","valor_m2_mediano","preco_mediano","area_mediana_m2","iqr_valor_m2","top_decil"],filteredRegions());\n  });\n  render();\n})();\n</script></body></html>\n\n'
COUNTY_GEOJSON = {'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'id': '53033', 'properties': {'name': 'King'}, 'geometry': {'type': 'MultiPolygon', 'coordinates': [[[[-122.39701267762678, 47.7782449706397], [-121.99142245642456, 47.775667791567905], [-121.58583223522234, 47.7773859109491], [-121.4566176514765, 47.78082214971149], [-121.11922401614015, 47.77996309002089], [-121.06538460624606, 47.71381549384493], [-121.1228133101331, 47.68546652405523], [-121.1335811921119, 47.64594977828777], [-121.11204542815427, 47.59698337592375], [-121.21613495394953, 47.56949346582465], [-121.30227800978008, 47.486164675836754], [-121.42790329953299, 47.420876139351385], [-121.43867118151181, 47.37190973698736], [-121.46738553345533, 47.367614438534375], [-121.43149259352593, 47.3349701702917], [-121.42790329953299, 47.28944000669006], [-121.34176024370242, 47.28084940978409], [-121.33817094970948, 47.24734608185081], [-121.36688530165301, 47.224151470204696], [-121.31663518575185, 47.21212463453634], [-121.30227800978008, 47.1391045608356], [-121.39559965359652, 47.13309114300142], [-121.38124247762477, 47.08756097939979], [-121.44584976949768, 47.084983800328], [-121.48892129741296, 47.115909949189486], [-121.58942152921529, 47.12020524764247], [-121.66120740907408, 47.159721993409924], [-121.68633246702467, 47.14941327712276], [-121.79401128681286, 47.17346694845948], [-121.84426140271401, 47.143399859288586], [-121.905279400594, 47.15370857557575], [-121.93758304653045, 47.138245501145], [-121.99860104441044, 47.17260788876888], [-122.04526186631865, 47.17174882907828], [-122.10986915819157, 47.20868839577395], [-122.14217280412802, 47.25679573844738], [-122.33599467974679, 47.25765479813797], [-122.4185484415844, 47.32036615555155], [-122.32522679776797, 47.34871512534124], [-122.32522679776797, 47.39166810987109], [-122.35394114971149, 47.44149357192571], [-122.38265550165501, 47.45094322852228], [-122.36111973769738, 47.48101031769317], [-122.39701267762678, 47.51537270531705], [-122.42213773557735, 47.57636594334942], [-122.34317326773267, 47.60901021159211], [-122.42931632356323, 47.65883567364673], [-122.37188761967619, 47.729278568275674], [-122.39701267762678, 47.7782449706397]]], [[[-122.52622726137261, 47.35902384162841], [-122.51187008540084, 47.44922510914108], [-122.47597714547145, 47.51107740686406], [-122.43290561755617, 47.46640630295302], [-122.44008420554205, 47.4165808408984], [-122.37188761967619, 47.3890909307993], [-122.4364949115491, 47.36589631915319], [-122.45444138151382, 47.343560767197665], [-122.51904867338672, 47.3332520509105], [-122.52622726137261, 47.35902384162841]]]]}}]}

def _zipcode_dominante(serie: pd.Series) -> str:
    moda = serie.astype("Int64").astype(str).mode()
    return moda.iat[0] if len(moda) else "não informado"

def _base_espacial(df: pd.DataFrame) -> pd.DataFrame:
    colunas = ["date", "price", "bedrooms", "bathrooms", "sqft_living", "lat", "long", "zipcode"]
    ausentes = set(colunas).difference(df.columns)
    if ausentes:
        raise KeyError(f"Colunas ausentes para o painel: {sorted(ausentes)}")
    base = df[colunas].dropna().copy()
    base = base[
        base["price"].gt(0) & base["sqft_living"].gt(0)
        & base["lat"].between(-90, 90) & base["long"].between(-180, 180)
    ]
    base["area_m2"] = base["sqft_living"] * 0.092903
    base["valor_m2"] = base["price"] / base["area_m2"]
    base["data_iso"] = pd.to_datetime(
        base["date"].astype(str), format="mixed", errors="raise"
    ).dt.strftime("%Y-%m-%d")
    return base

def agregar_valor_m2_h3(
    df: pd.DataFrame,
    resolucao: int = H3_RESOLUTION,
    minimo_vendas: int = H3_MIN_SALES,
) -> tuple[pd.DataFrame, dict]:
    """Resume o valor por m² em H3; indicador restrito à análise exploratória."""
    base = _base_espacial(df)
    base["h3"] = [h3.latlng_to_cell(lat, lon, resolucao) for lat, lon in zip(base["lat"], base["long"])]
    regioes = (
        base.groupby("h3", as_index=False)
        .agg(
            vendas=("price", "size"),
            valor_m2_mediano=("valor_m2", "median"),
            valor_m2_q1=("valor_m2", lambda s: s.quantile(0.25)),
            valor_m2_q3=("valor_m2", lambda s: s.quantile(0.75)),
            preco_mediano=("price", "median"),
            area_mediana_m2=("area_m2", "median"),
            zipcode_dominante=("zipcode", _zipcode_dominante),
        )
    )
    celulas_ocupadas = len(regioes)
    regioes = regioes[regioes["vendas"].ge(minimo_vendas)].copy()
    regioes[["latitude_centro", "longitude_centro"]] = pd.DataFrame(
        [h3.cell_to_latlng(celula) for celula in regioes["h3"]], index=regioes.index
    )
    regioes["iqr_valor_m2"] = regioes["valor_m2_q3"] - regioes["valor_m2_q1"]
    limite_topo = float(regioes["valor_m2_mediano"].quantile(0.90))
    regioes["top_decil"] = regioes["valor_m2_mediano"].ge(limite_topo)
    regioes = regioes.sort_values("valor_m2_mediano", ascending=False, ignore_index=True)
    resumo = {
        "metrica": "mediana do preço por m²",
        "resolucao_h3": resolucao,
        "minimo_vendas_por_celula": minimo_vendas,
        "vendas_validas": int(len(base)),
        "vendas_mapeadas": int(regioes["vendas"].sum()),
        "cobertura_percentual": float(100 * regioes["vendas"].sum() / len(base)),
        "celulas_ocupadas": int(celulas_ocupadas),
        "celulas_exibidas": int(len(regioes)),
        "celulas_top_decil": int(regioes["top_decil"].sum()),
        "limite_top_decil_usd_m2": limite_topo,
        "escala_visual_p05": float(regioes["valor_m2_mediano"].quantile(0.05)),
        "escala_visual_p95": float(regioes["valor_m2_mediano"].quantile(0.95)),
    }
    return regioes, resumo

def _geojson_h3(regioes: pd.DataFrame) -> dict:
    features = []
    for celula in regioes["h3"]:
        contorno = [[lon, lat] for lat, lon in h3.cell_to_boundary(celula)]
        contorno.append(contorno[0])
        features.append({
            "type": "Feature", "id": celula, "properties": {"h3": celula},
            "geometry": {"type": "Polygon", "coordinates": [contorno]},
        })
    return {"type": "FeatureCollection", "features": features}

def criar_mapa_interativo(regioes: pd.DataFrame, resumo: dict):
    """Cria uma prévia H3 compacta para a saída do notebook."""
    fig = px.choropleth_mapbox(
        regioes, geojson=_geojson_h3(regioes), locations="h3",
        featureidkey="properties.h3", color="valor_m2_mediano",
        color_continuous_scale=ATLANTIC,
        range_color=(resumo["escala_visual_p05"], resumo["escala_visual_p95"]),
        opacity=0.82, mapbox_style="carto-positron",
        center={"lat": float(regioes["latitude_centro"].median()), "lon": float(regioes["longitude_centro"].median())},
        zoom=8, hover_name="zipcode_dominante",
        hover_data={
            "h3": False, "valor_m2_mediano": ":,.0f", "vendas": ":,",
            "preco_mediano": ":,.0f", "area_mediana_m2": ":.1f",
            "iqr_valor_m2": ":,.0f", "top_decil": True,
            "latitude_centro": False, "longitude_centro": False,
        },
        labels={
            "zipcode_dominante": "ZIP predominante", "valor_m2_mediano": "Mediana (US$/m²)",
            "vendas": "Vendas", "preco_mediano": "Preço mediano (US$)",
            "area_mediana_m2": "Área mediana (m²)", "iqr_valor_m2": "IQR (US$/m²)",
            "top_decil": "Decil superior",
        },
        title="Valor residencial por metro quadrado — King County",
    )
    fig.update_traces(marker_line_width=0.45, marker_line_color="#f8fafc")
    topo = regioes[regioes["top_decil"]]
    fig.add_trace(go.Choroplethmapbox(
        geojson=_geojson_h3(topo), locations=topo["h3"], z=np.ones(len(topo)),
        featureidkey="properties.h3", colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
        showscale=False, marker={"line": {"color": "#b08d57", "width": 1.8}}, hoverinfo="skip",
    ))
    fig.update_layout(
        height=680, margin={"l": 0, "r": 0, "t": 70, "b": 0},
        font={"family": "Arial", "color": "#172033"},
        coloraxis_colorbar={"title": "US$/m²", "tickformat": "$,.0f"},
        title={"x": 0.02, "font": {"size": 22}},
    )
    return fig

def _regioes_painel(regioes: pd.DataFrame) -> list[dict]:
    registros = []
    for row in regioes.itertuples(index=False):
        contorno = [[lon, lat] for lat, lon in h3.cell_to_boundary(row.h3)]
        contorno.append(contorno[0])
        registros.append({
            "h3": row.h3, "vendas": int(row.vendas),
            "valor_m2_mediano": float(row.valor_m2_mediano),
            "preco_mediano": float(row.preco_mediano),
            "area_mediana_m2": float(row.area_mediana_m2),
            "zipcode_dominante": str(row.zipcode_dominante),
            "iqr_valor_m2": float(row.iqr_valor_m2), "top_decil": bool(row.top_decil),
            "boundary": contorno,
        })
    return registros

def _transacoes_painel(df: pd.DataFrame) -> list[dict]:
    base = _base_espacial(df)
    return [{
        "lat": float(row.lat), "lon": float(row.long), "price": float(row.price),
        "valor_m2": float(row.valor_m2), "zipcode": str(int(row.zipcode)),
        "date": row.data_iso, "bedrooms": int(row.bedrooms),
        "bathrooms": float(row.bathrooms), "sqft_living": int(row.sqft_living),
    } for row in base.itertuples(index=False)]

def salvar_painel_geoespacial(df: pd.DataFrame, regioes: pd.DataFrame) -> Path:
    pagina = PANEL_TEMPLATE
    substituicoes = {
        "__REGIONS_JSON__": _regioes_painel(regioes),
        "__TRANSACTIONS_JSON__": _transacoes_painel(df),
        "__COUNTY_JSON__": COUNTY_GEOJSON,
    }
    for marcador, dados in substituicoes.items():
        pagina = pagina.replace(marcador, json.dumps(dados, ensure_ascii=False, separators=(",", ":")))
    pendentes = [marcador for marcador in substituicoes if marcador in pagina]
    if pendentes:
        raise RuntimeError(f"Marcadores não substituídos: {pendentes}")
    MAPS_DIR.mkdir(parents=True, exist_ok=True)
    PANEL_PATH.write_text(pagina, encoding="utf-8")
    return PANEL_PATH

def gerar_mapa_valor_m2(df: pd.DataFrame):
    regioes, resumo = agregar_valor_m2_h3(df)
    MAPS_DIR.mkdir(parents=True, exist_ok=True)
    regioes.to_csv(MAPS_DIR / "regioes_h3_valor_m2.csv", index=False)
    (MAPS_DIR / "resumo_mapa_valor_m2.json").write_text(
        json.dumps(resumo, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    painel = salvar_painel_geoespacial(df, regioes)
    return regioes, resumo, criar_mapa_interativo(regioes, resumo), painel

In [ ]:
regioes_h3, resumo_mapa, mapa_interativo, painel_geoespacial = gerar_mapa_valor_m2(df_corr)
display(mapa_interativo)
print("Painel completo gerado em:", painel_geoespacial)
if em_colab:
    def baixar_painel():
        from google.colab import files
        files.download(str(painel_geoespacial))
    print("Opcional: execute baixar_painel() para obter o painel HTML completo.")
else:
    display(FileLink(str(painel_geoespacial), result_html_prefix="Painel completo: "))
display(pd.DataFrame([resumo_mapa]).style.format({"cobertura_percentual": "{:.1f}", "limite_top_decil_usd_m2": "{:,.0f}"}))
display(regioes_h3.head(10)[["zipcode_dominante", "valor_m2_mediano", "vendas", "preco_mediano"]].style.format({"valor_m2_mediano": "US$ {:,.0f}", "preco_mediano": "US$ {:,.0f}"}))


O painel conserva **20.014 das 21.613 vendas**, cobertura de **92,6%** da base, em **290 células H3** com pelo menos 20 negócios. O decil superior começa em aproximadamente **US\$ 3.824/m²** e reúne 29 células. Essa concentração mostra que a localização opera em microrregiões, não apenas por limites administrativos; por isso, o painel serve à seleção de comparáveis e `zipcode`, latitude e longitude permanecem entre os preditores. O valor por m² não entra no modelo porque é calculado a partir do próprio preço e causaria vazamento do alvo.


## **2.3 Análise textual crítica: como a EDA orienta a modelagem**

Os resultados visuais e estatísticos formam uma cadeia de decisão. A diferença entre média e mediana, somada à assimetria de **4,024** e à curtose excedente de **34,586**, mostra que o preço original é dominado por uma cauda de transações muito caras. Por isso, o projeto compara a escala original com `log1p(price)` em vez de presumir que uma única transformação seja superior.

`sqft_living` e `grade` apresentam associação forte com o preço, mas os gráficos também revelam curvaturas, faixas e dispersão crescente. Isso justifica manter a Regressão Linear como baseline interpretável e confrontá-la com modelos de árvores capazes de representar interações. A diferença entre Pearson e Spearman em `lat`, junto ao painel H3, confirma que localização contém estrutura não estritamente linear; `zipcode`, latitude e longitude permanecem na modelagem, enquanto `preco_por_m2` fica restrito à análise por derivar do próprio alvo.

A matriz expõe redundância entre medidas de área. Essa evidência conduz ao VIF e à retirada de variáveis estruturalmente equivalentes antes do ajuste linear. Por fim, as 176 propriedades revendidas exigem divisão agrupada por `id`: manter o mesmo imóvel em treino e teste produziria uma avaliação artificialmente otimista. Assim, **transformação do alvo, comparação de modelos, controle de multicolinearidade e separação agrupada decorrem diretamente da EDA**, não de escolhas posteriores desconectadas dos dados.


# **3. Fase 2 — Tratamento e Limpeza (Data Prep)**

Ausências e valores extremos são tratados de acordo com sua natureza, preservando observações raras que continuam plausíveis no domínio imobiliário.


In [ ]:
def limpar_dados(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    dados = df.drop_duplicates().copy()
    dados["date"] = pd.to_datetime(dados["date"], format="%Y%m%dT%H%M%S")

    mediana_sqft_above = float(dados["sqft_above"].median())
    nulos_imputados = int(dados["sqft_above"].isna().sum())
    dados["sqft_above"] = dados["sqft_above"].fillna(mediana_sqft_above)

    anomalia = _quartos_incompativeis_com_area(dados)
    comparaveis = dados.loc[
        ~anomalia
        & dados["sqft_living"].between(1_400, 1_800)
        & dados["bathrooms"].between(1.5, 2.0),
        "bedrooms",
    ]
    quartos_substitutos = float(comparaveis.median()) if len(comparaveis) else 3.0
    dados.loc[anomalia, "bedrooms"] = quartos_substitutos

    relatorio = {
        "duplicatas_removidas": int(len(df) - len(dados)),
        "nulos_sqft_above_imputados": nulos_imputados,
        "mediana_sqft_above": mediana_sqft_above,
        "anomalias_bedrooms_corrigidas": int(anomalia.sum()),
        "imoveis_comparaveis": int(len(comparaveis)),
        "mediana_quartos_comparaveis": quartos_substitutos,
    }
    return dados, relatorio

def plot_boxplots(df: pd.DataFrame):
    colunas = ["price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot"]
    fig, axes = plt.subplots(1, len(colunas), figsize=(15, 5))
    for ax, coluna in zip(axes, colunas):
        sns.boxplot(y=df[coluna], ax=ax, color="#bfdbfe")
        ax.set(title=coluna, xlabel="", ylabel="")
    return _salvar(fig, "04_boxplots_outliers.png")

df_limpo, relatorio_limpeza = limpar_dados(df_raw)
relatorio_qualidade = {
    "auditoria_inicial": auditoria, "tratamento": relatorio_limpeza,
    "quartos_11_preservados": int(df_limpo["bedrooms"].eq(11).sum()),
    "quartos_33_remanescentes": int(df_limpo["bedrooms"].eq(33).sum()),
}
(ROOT / "outputs/relatorio_qualidade_dados.json").write_text(json.dumps(relatorio_qualidade, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.Series(relatorio_limpeza, name="resultado").to_frame())
plot_boxplots(df_raw); plt.show()


A auditoria encontrou **zero duplicata integral**; portanto, `drop_duplicates()` não eliminou nenhuma venda. Os IDs repetidos foram mantidos porque têm data e preço próprios e representam revendas, não cópias da mesma observação. O risco de vazamento entre essas transações é controlado posteriormente pelo agrupamento por `id`.

As **duas ausências de `sqft_above`** foram substituídas pela **mediana de 1.560 ft²**. A escolha foi explicitamente orientada pela distribuição: a variável tem assimetria positiva de aproximadamente **1,447**, média de **1.788 ft²** e máximo de **9.410 ft²**. A média seria deslocada para cima pelos imóveis muito grandes; a mediana isola melhor essa distorção e representa o centro típico da amostra. A imputação pela média seria mais adequada se a distribuição fosse aproximadamente simétrica, o que os valores e os boxplots não sustentam neste caso.

Os boxplots foram usados para distinguir erro de raridade. O imóvel de **11 quartos e 3.000 ft² foi preservado**, pois mantém área plausível por quarto. Somente o registro de **33 quartos em 1.620 ft²** violou a regra física adotada e foi corrigido para **três quartos**, mediana de 2.328 comparáveis. Mansões, terrenos extensos e vendas de alto padrão foram mantidos: uma winsorização ampla reduziria justamente a variação que o modelo precisa aprender e distorceria os coeficientes da Regressão Linear. A comparação posterior com árvores mais robustas a extremos permite avaliar essa sensibilidade sem apagar transações legítimas.


# **4. Fase 3 — Feature Engineering (Colunas Calculadas)**

As novas variáveis representam idade, reforma, porão, sazonalidade e escala do terreno usando apenas informação disponível antes da venda.


In [ ]:
def criar_features(df: pd.DataFrame) -> pd.DataFrame:
    dados = df.copy()
    ano_venda = dados["date"].dt.year
    idade_imovel = (ano_venda - dados["yr_built"]).clip(lower=0)
    area_m2 = dados["sqft_living"] * 0.092903
    dados = dados.assign(
        ano_venda=ano_venda,
        mes_venda=dados["date"].dt.month,
        idade_imovel=idade_imovel,
        foi_reformado=dados["yr_renovated"].gt(0).astype("int8"),
        tem_porao=dados["sqft_basement"].gt(0).astype("int8"),
        log_sqft_lot=np.log1p(dados["sqft_lot"]),
        log_sqft_lot15=np.log1p(dados["sqft_lot15"]),
        area_m2=area_m2,
    )
    if TARGET in dados:
        dados["preco_por_m2"] = dados[TARGET] / area_m2
    return dados

df = criar_features(df_limpo)
salvar_csv(df, PROCESSED_DATA)
features_novas = ["ano_venda", "mes_venda", "idade_imovel", "foi_reformado", "tem_porao", "log_sqft_lot", "area_m2", "preco_por_m2"]
display(df[features_novas].describe().T.style.format("{:,.2f}"))
print("Idades negativas:", int(df["idade_imovel"].lt(0).sum()))
print("Casos temporais limitados a zero:", int((df["yr_built"].gt(df["date"].dt.year) & df["idade_imovel"].eq(0)).sum()))


`idade_imovel` traduz `yr_built` para a idade efetiva na venda; `foi_reformado` e `tem_porao` transformam códigos em indicadores diretamente interpretáveis; mês e ano preservam o contexto temporal; `log_sqft_lot` reduz a compressão visual e numérica causada pelos terrenos extremos. As **12 idades negativas foram limitadas a zero**, leitura compatível com venda na planta ou conclusão no mesmo período, sem inventar uma idade positiva.

`preco_por_m2` permanece apenas na base analítica e no painel espacial. Como seu numerador é `price`, incluí-lo entre os preditores revelaria diretamente a variável-alvo e produziria **vazamento de dados**. A engenharia usada para inferência — inclusive data, idade, reforma, porão e log do terreno — passa a integrar o próprio pipeline salvo, garantindo a mesma transformação em treino e em novos imóveis.


# **5. Fase 4 — Preparação para Modelagem**

O **VIF — Fator de Inflação da Variância** verifica quanto a variância do coeficiente de uma variável aumenta porque ela pode ser explicada pelas demais. Para cada preditor, aplica-se `VIF = 1 / (1 − R² auxiliar)`: valor próximo de 1 indica baixa redundância; acima de 5 pede atenção; acima de 10 costuma indicar colinearidade severa.

| Faixa de VIF | Leitura | Consequência prática |
|---|---|---|
| próximo de 1 | pouca redundância | coeficiente mais estável |
| entre 1 e 5 | dependência moderada | geralmente aceitável |
| entre 5 e 10 | atenção | interpretação individual perde precisão |
| acima de 10 | redundância forte | revisar ou retirar variáveis equivalentes |

VIF alto não prova que o modelo prevê mal; ele compromete principalmente a interpretação isolada dos coeficientes. Neste projeto, a identidade entre área total, área acima do solo e porão exige redução consciente de redundância.

Depois do VIF, a divisão usa `GroupShuffleSplit` por `id`. Assim, revendas do mesmo imóvel permanecem no mesmo conjunto, evitando que o teste receba informação quase duplicada do treino.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

def calcular_vif(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    matriz = SimpleImputer(strategy="median").fit_transform(df[colunas])
    valores = []
    for indice, coluna in enumerate(colunas):
        demais = np.delete(matriz, indice, axis=1)
        r2 = LinearRegression().fit(demais, matriz[:, indice]).score(demais, matriz[:, indice])
        valores.append({"variavel": coluna, "VIF": 1 / max(1 - r2, 1e-12)})
    return pd.DataFrame(valores).sort_values("VIF", ascending=False, ignore_index=True)

def preparar_preditores(df: pd.DataFrame) -> pd.DataFrame:
    X = df[MODEL_FEATURES].copy()
    X["zipcode"] = X["zipcode"].astype(str)
    return X

def separar_brutos_alvo_grupos(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    X_bruto = df.drop(columns=[TARGET, "id"]).copy()
    return X_bruto, df[TARGET].copy(), df["id"].copy()

NUMERIC_FEATURES = ["bedrooms","bathrooms","sqft_living","floors","waterfront","view","condition","grade","sqft_basement","lat","long","sqft_living15","log_sqft_lot","idade_imovel","foi_reformado","tem_porao","mes_venda","ano_venda"]
CATEGORICAL_FEATURES = ["zipcode"]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
VIF_BEFORE = ["bedrooms","bathrooms","sqft_living","floors","waterfront","view","condition","grade","sqft_above","sqft_basement","lat","long","sqft_living15","log_sqft_lot","log_sqft_lot15","idade_imovel"]
VIF_AFTER = [c for c in VIF_BEFORE if c not in {"sqft_above", "log_sqft_lot15"}]

vif_antes, vif_depois = calcular_vif(df, VIF_BEFORE), calcular_vif(df, VIF_AFTER)
vif_antes.to_csv(ROOT / "outputs/vif_antes.csv", index=False)
vif_depois.to_csv(ROOT / "outputs/vif_depois.csv", index=False)
X_model = preparar_preditores(df)
X, y, grupos = separar_brutos_alvo_grupos(df_limpo)
idx_train, idx_test = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(X, y, grupos))
X_train, X_test, y_train, y_test = X.iloc[idx_train], X.iloc[idx_test], y.iloc[idx_train], y.iloc[idx_test]
grupos_train, grupos_test = grupos.iloc[idx_train], grupos.iloc[idx_test]
display(vif_antes.head(8).style.format({"VIF": "{:.2f}"}))
display(vif_depois.head(8).style.format({"VIF": "{:.2f}"}))
print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")
print("IDs presentes nos dois conjuntos:", len(set(grupos_train) & set(grupos_test)))


A identidade `sqft_living = sqft_above + sqft_basement` produzia VIF extremo. A retirada de `sqft_above` preservou a área total e a informação de porão; `log_sqft_lot15` também saiu por redundância. O **maior VIF remanescente ficou em aproximadamente 6,57**: ainda merece cautela interpretativa, mas não configura a dependência quase determinística observada antes.

`zipcode` é nominal e não possui distância numérica legítima entre seus códigos; por isso, recebe **One-Hot Encoding**, com `handle_unknown="ignore"` para que um código não visto não interrompa a inferência. As variáveis numéricas passam por imputação pela mediana e `StandardScaler`. Essas operações ficam dentro de `ColumnTransformer` e `Pipeline`: em cada fold e no holdout, o `fit` ocorre somente no subconjunto de treino, enquanto a validação e o teste recebem apenas `transform`. Essa estrutura é a forma segura e reproduzível de demonstrar `fit_transform` no treino e `transform` fora dele, sem vazamento.

A separação gerou **17.289 registros de treino**, **4.324 de teste** e **zero ID compartilhado**. O objeto de entrada preserva as colunas brutas anteriores à venda; a engenharia de atributos, o encoding, a escala e o modelo serão encapsulados no mesmo pipeline para que a inferência não dependa de reconstrução manual.


# **6. Fase 5 — Modelagem, Validação e Diagnóstico de Overfitting**

Regressão estima um **alvo contínuo** — aqui, o preço do imóvel. Conforme trabalhado na Semana 11, o resíduo é `preço real − preço previsto`; os modelos diferem na forma de reduzir e distribuir esses erros.

### **6.1 Por que usar cinco folds?**

O conjunto de treino é dividido em cinco partes. Em cada rodada, quatro partes treinam o modelo e a quinta o valida; o processo se repete até todas terem servido como validação. Com os grupos por `id`, cada imóvel permanece inteiro em um único fold. A média mede desempenho esperado; o desvio entre folds mede estabilidade.

| Modelo | Como representa o preço | Vantagem | Limitação | Papel no projeto |
|---|---|---|---|---|
| Baseline pela média | repete a média do treino para todos os imóveis | referência mínima e transparente | ignora todas as características | comprovar se os modelos aprendem algo útil |
| Regressão Linear | soma efeitos lineares | simples e interpretável | não captura bem curvaturas e interações | **modelo base obrigatório** |
| HistGradientBoosting | combina árvores sequenciais | aprende não linearidades e interações | menor transparência direta | candidato flexível na escala original |
| HistGradientBoosting Log | modela `log1p(price)` e corrige a volta à escala | reduz influência da cauda de preços altos | interpretação menos imediata | candidato adequado à assimetria observada |

O baseline ingênuo ancora o R²: prever apenas a média tende a produzir R² próximo de zero. A Regressão Linear atende ao modelo base da rubrica e verifica quanto se ganha com relações lineares; os dois boostings constituem o diferencial comparativo.

O campeão é escolhido pelo **menor RMSE médio nos folds**, antes de consultar o teste. O RMSE recebe esse papel porque penaliza mais fortemente erros grandes, relevantes na precificação de imóveis de alto valor. MAE, R² e MAPE funcionam como critérios de consistência: a escolha não deve reduzir RMSE às custas de deterioração material nas demais métricas.


In [ ]:
from collections import OrderedDict
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

class LogSmearingRegressor(RegressorMixin, BaseEstimator):
    """Regressão em log com correção de retransformação aprendida no treino."""

    def __init__(self, regressor):
        self.regressor = regressor

    def fit(self, X, y):
        y = np.asarray(y)
        if np.any(y < 0):
            raise ValueError("O alvo deve ser não negativo para a transformação log1p.")
        y_log = np.log1p(y)
        self.regressor_ = clone(self.regressor).fit(X, y_log)
        residuos_log = y_log - self.regressor_.predict(X)
        self.smearing_factor_ = float(np.mean(np.exp(residuos_log)))
        return self

    def predict(self, X):
        check_is_fitted(self, ["regressor_", "smearing_factor_"])
        previsao = np.exp(self.regressor_.predict(X)) * self.smearing_factor_ - 1
        return np.maximum(previsao, 0)


class RawFeatureTransformer(TransformerMixin, BaseEstimator):
    """Aprende a correção contextual e deriva preditores a partir do esquema bruto."""

    def __init__(self, feature_names=tuple(MODEL_FEATURES)):
        self.feature_names = feature_names

    @staticmethod
    def _normalizar_data(df: pd.DataFrame) -> pd.DataFrame:
        dados = df.copy()
        if "date" not in dados:
            raise ValueError("A coluna bruta 'date' é obrigatória para a inferência.")
        if not pd.api.types.is_datetime64_any_dtype(dados["date"]):
            dados["date"] = pd.to_datetime(
                dados["date"].astype(str), format="%Y%m%dT%H%M%S", errors="raise"
            )
        return dados

    def fit(self, X, y=None):
        dados = self._normalizar_data(X)
        anomalia = _quartos_incompativeis_com_area(dados)
        comparaveis = dados.loc[
            ~anomalia
            & dados["sqft_living"].between(1_400, 1_800)
            & dados["bathrooms"].between(1.5, 2.0),
            "bedrooms",
        ]
        self.quartos_substitutos_ = float(comparaveis.median()) if len(comparaveis) else 3.0
        self.feature_names_in_ = np.asarray(dados.columns, dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        return self

    def transform(self, X):
        check_is_fitted(self, ["quartos_substitutos_", "feature_names_in_"])
        dados = self._normalizar_data(X)
        anomalia = _quartos_incompativeis_com_area(dados)
        dados.loc[anomalia, "bedrooms"] = self.quartos_substitutos_
        return preparar_preditores(criar_features(dados)).loc[:, list(self.feature_names)]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_in_")
        return np.asarray(self.feature_names, dtype=object)

def construir_preprocessador(numericas: list[str], categoricas: list[str]) -> ColumnTransformer:
    numerico = Pipeline(
        [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    )
    try:
        categorico = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        categorico = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([("num", numerico, numericas), ("cat", categorico, categoricas)])

def _hgb(random_state: int) -> HistGradientBoostingRegressor:
    return HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.06,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5.0,
        early_stopping=True,
        random_state=random_state,
    )

def modelos_candidatos(random_state: int = 42) -> OrderedDict:
    return OrderedDict(
        {
            "Baseline pela média": DummyRegressor(strategy="mean"),
            "Regressão Linear": LinearRegression(),
            "HistGradientBoosting": _hgb(random_state),
            "HistGradientBoosting Log": LogSmearingRegressor(_hgb(random_state)),
        }
    )

def criar_pipeline(preprocessador, modelo) -> Pipeline:
    return Pipeline(
        [
            ("engenharia_atributos", RawFeatureTransformer()),
            ("preprocessador", clone(preprocessador)),
            ("modelo", clone(modelo)),
        ]
    )

def calcular_metricas(y_true, y_pred) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }

def validar_candidatos(modelos, preprocessador, X, y, grupos, folds=5, random_state=42):
    try:
        cv = GroupKFold(n_splits=folds, shuffle=True, random_state=random_state)
    except TypeError:
        cv = GroupKFold(n_splits=folds)
    linhas = []
    scoring = {
        "mae": "neg_mean_absolute_error",
        "mse": "neg_mean_squared_error",
        "r2": "r2",
        "mape": "neg_mean_absolute_percentage_error",
    }
    for nome, modelo in modelos.items():
        scores = cross_validate(
            criar_pipeline(preprocessador, modelo),
            X,
            y,
            groups=grupos,
            cv=cv,
            scoring=scoring,
            n_jobs=N_JOBS,
        )
        for fold in range(folds):
            linhas.append(
                {
                    "modelo": nome,
                    "fold": fold + 1,
                    "MAE": -scores["test_mae"][fold],
                    "RMSE": np.sqrt(-scores["test_mse"][fold]),
                    "R2": scores["test_r2"][fold],
                    "MAPE": -scores["test_mape"][fold],
                }
            )
    por_fold = pd.DataFrame(linhas)
    resumo = (
        por_fold.groupby("modelo", sort=False)
        .agg(
            MAE_medio=("MAE", "mean"),
            MAE_desvio=("MAE", "std"),
            RMSE_medio=("RMSE", "mean"),
            RMSE_desvio=("RMSE", "std"),
            R2_medio=("R2", "mean"),
            R2_desvio=("R2", "std"),
            MAPE_medio=("MAPE", "mean"),
            MAPE_desvio=("MAPE", "std"),
        )
        .reset_index()
    )
    return por_fold, resumo.sort_values("RMSE_medio", ignore_index=True)

def plot_validacao_cruzada(por_fold: pd.DataFrame):
    """Exibe o resultado de cada fold e o ganho relativo ao palpite médio."""
    ordem = por_fold.groupby("modelo")["RMSE"].mean().sort_values().index.tolist()
    baseline = por_fold.loc[
        por_fold["modelo"].eq("Baseline pela média"), ["fold", "RMSE"]
    ].rename(columns={"RMSE": "RMSE_baseline"})
    comparacao = por_fold.merge(baseline, on="fold", how="left")
    comparacao["reducao_RMSE"] = 100 * (
        comparacao["RMSE_baseline"] - comparacao["RMSE"]
    ) / comparacao["RMSE_baseline"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    cores = ["#21485f", "#5e959f", "#b08d57", "#9aa8b5"]
    for posicao, (nome, cor) in enumerate(zip(ordem, cores)):
        valores = por_fold.loc[por_fold["modelo"].eq(nome), "RMSE"].to_numpy()
        deslocamento = np.linspace(-0.07, 0.07, len(valores))
        axes[0].scatter(np.full(len(valores), posicao) + deslocamento, valores, color=cor, s=46, alpha=.85)
        axes[0].scatter(posicao, valores.mean(), marker="D", color="#172033", s=50, zorder=3)

    aprendidos = [nome for nome in ordem if nome != "Baseline pela média"]
    for posicao, (nome, cor) in enumerate(zip(aprendidos, cores)):
        valores = comparacao.loc[comparacao["modelo"].eq(nome), "reducao_RMSE"].to_numpy()
        deslocamento = np.linspace(-0.07, 0.07, len(valores))
        axes[1].scatter(np.full(len(valores), posicao) + deslocamento, valores, color=cor, s=46, alpha=.85)
        axes[1].scatter(posicao, valores.mean(), marker="D", color="#172033", s=50, zorder=3)

    axes[0].set(title="RMSE observado em cada fold", ylabel="RMSE (US$)", xlabel="")
    axes[0].set_xticks(range(len(ordem)), ordem, rotation=14)
    axes[1].axhline(0, color="#9aa8b5", lw=1)
    axes[1].set(title="Redução do RMSE frente ao baseline", ylabel="Redução (%)", xlabel="")
    axes[1].set_xticks(range(len(aprendidos)), aprendidos, rotation=14)
    fig.suptitle("Estabilidade da validação cruzada — pontos: folds; losangos: médias", y=1.02)
    return _salvar(fig, "05_validacao_cruzada_modelos.png")

preprocessador = construir_preprocessador(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
modelos = modelos_candidatos(SEED)
cv_folds, cv_resumo = validar_candidatos(modelos, preprocessador, X_train, y_train, grupos_train, CV_FOLDS, SEED)
cv_folds.to_csv(ROOT / "outputs/cv_modelos_folds.csv", index=False)
cv_resumo.to_csv(ROOT / "outputs/cv_modelos_resumo.csv", index=False)
vitorias_por_fold = cv_folds.loc[
    cv_folds.groupby("fold")["RMSE"].idxmin(), ["fold", "modelo", "RMSE"]
].sort_values("fold", ignore_index=True)
vitorias_por_fold.to_csv(ROOT / "outputs/vitorias_por_fold.csv", index=False)
campeao_nome = cv_resumo.iloc[0]["modelo"]
display(cv_resumo.style.format({"MAE_medio": "US$ {:,.0f}", "RMSE_medio": "US$ {:,.0f}", "R2_medio": "{:.4f}", "MAPE_medio": "{:.2%}"}))
print("Modelo selecionado sem consultar o teste:", campeao_nome)
display(vitorias_por_fold.style.format({"RMSE": "US$ {:,.0f}"}))
plot_validacao_cruzada(cv_folds); plt.show()


## **6.2 Escolha técnica do modelo**

O **HistGradientBoosting Log** foi escolhido por uma sequência de evidências, e não apenas por ocupar a primeira linha da tabela:

1. **critério definido antes do teste:** apresentou o menor RMSE médio, aproximadamente **US\$ 119.154**;
2. **consistência entre métricas:** também obteve o menor MAE (**US\$ 63.780**) e MAPE (**11,99%**) e o maior R² (**0,8933**);
3. **estabilidade:** venceu pelo RMSE nos **cinco folds**, embora cada fold contenha imóveis diferentes;
4. **ganho econômico frente ao palpite mínimo:** contra o baseline pela média, reduziu o MAE em **72,6%** e o RMSE em **67,4%**;
5. **coerência com a distribuição:** a transformação log reduziu a assimetria de **4,024 para 0,428**, enquanto o boosting capturou relações não lineares entre localização, área e padrão construtivo.

A alternativa na escala original ficou em segundo lugar. A diferença indica que não basta usar árvores: controlar a cauda do alvo melhora o equilíbrio entre imóveis usuais e transações muito caras. A Regressão Linear permanece valiosa como referência interpretável, mas seu RMSE médio de aproximadamente **US\$ 160.244** mostra a limitação de representar esse mercado apenas por efeitos aditivos.

O holdout continuou intocado durante essa decisão. Essa ordem evita escolher o modelo por um resultado casual no teste e preserva o teste como verificação final de generalização.


# **7. Fase 6 — Avaliação, Interpretação e Versionamento do Modelo**

## **7.1 Métricas, teste isolado e diagnóstico de generalização**

O teste permaneceu intocado durante a escolha. A comparação de treino e teste completa o diagnóstico de overfitting iniciado na Fase 5: diferenças pequenas indicam comportamento estável; deterioração acentuada sugere sobreajuste.

| Indicador | Conceito básico | Unidade | Interpretação neste projeto |
|---|---|---|---|
| MAE | média de `|real − previsto|` | dólares | erro monetário típico, fácil de comunicar |
| MSE | média de `(real − previsto)²` | dólares² | base matemática do RMSE; amplia fortemente erros grandes, mas tem unidade pouco intuitiva |
| RMSE | raiz do MSE | dólares | mantém a unidade do preço e enfatiza desvios severos |
| R² | redução da variabilidade residual em relação à média | proporção | capacidade de explicar a variação dos preços; **não é percentual de acerto** |
| MAPE | média de `|erro / real|` | percentual | erro relativo médio; útil porque todos os preços são positivos |

A tabela seguinte apresenta essas métricas em treino e teste para os quatro candidatos; o gráfico de RMSE e R² torna visível qualquer separação excessiva entre os conjuntos.


In [ ]:
def avaliar_holdout(modelos, preprocessador, X_train, X_test, y_train, y_test):
    ajustados, linhas = {}, []
    for nome, modelo in modelos.items():
        pipeline = criar_pipeline(preprocessador, modelo).fit(X_train, y_train)
        ajustados[nome] = pipeline
        for conjunto, X, y in [("Treino", X_train, y_train), ("Teste", X_test, y_test)]:
            linhas.append(
                {"modelo": nome, "conjunto": conjunto, **calcular_metricas(y, pipeline.predict(X))}
            )
    return ajustados, pd.DataFrame(linhas)

def diagnosticar_generalizacao(resultados: pd.DataFrame) -> pd.DataFrame:
    linhas = []
    for modelo, grupo in resultados.groupby("modelo", sort=False):
        tabela = grupo.set_index("conjunto")
        linhas.append(
            {
                "modelo": modelo,
                "delta_R2": tabela.loc["Treino", "R2"] - tabela.loc["Teste", "R2"],
                "razao_RMSE_teste_treino": tabela.loc["Teste", "RMSE"] / tabela.loc["Treino", "RMSE"],
                "razao_MAE_teste_treino": tabela.loc["Teste", "MAE"] / tabela.loc["Treino", "MAE"],
            }
        )
    return pd.DataFrame(linhas)

def plot_generalizacao(resultados: pd.DataFrame):
    base = resultados[resultados["conjunto"].isin(["Treino", "Teste"])]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    sns.barplot(data=base, x="modelo", y="RMSE", hue="conjunto", ax=axes[0])
    sns.barplot(data=base, x="modelo", y="R2", hue="conjunto", ax=axes[1])
    axes[0].set(title="RMSE: treino × teste", xlabel="", ylabel="RMSE (US$)")
    axes[1].set(title="R²: treino × teste", xlabel="", ylabel="R²")
    for ax in axes:
        ax.tick_params(axis="x", rotation=13)
    return _salvar(fig, "06_generalizacao_modelos.png")

ajustados, resultados = avaliar_holdout(modelos, preprocessador, X_train, X_test, y_train, y_test)
diagnostico = diagnosticar_generalizacao(resultados)
resultados.to_csv(ROOT / "outputs/metricas_modelos.csv", index=False)
diagnostico.to_csv(ROOT / "outputs/diagnostico_generalizacao.csv", index=False)
display(resultados.style.format({"MAE": "US$ {:,.0f}", "MSE": "{:,.0f}", "RMSE": "US$ {:,.0f}", "R2": "{:.4f}", "MAPE": "{:.2%}"}))
display(diagnostico.style.format({"delta_R2": "{:.4f}", "razao_RMSE_teste_treino": "{:.3f}", "razao_MAE_teste_treino": "{:.3f}"}))
plot_generalizacao(resultados); plt.show()


No holdout, o campeão obteve **MAE de US\$ 64.720**, **MSE de aproximadamente US\$² 14,58 bilhões**, **RMSE de US\$ 120.736**, **R² de 0,8956** e **MAPE de 11,95%**. O MAE traduz o resultado para o negócio: uma estimativa individual se desvia, em média, cerca de **US\$ 64,7 mil**, ou **14,4% da mediana de US\$ 450 mil**. O RMSE quase duas vezes maior mostra que uma parcela menor de imóveis concentra erros muito superiores ao típico.

No campeão, o R² passa de **0,9174 no treino para 0,8956 no teste**, diferença de **0,0217**; o RMSE passa de **US\$ 105.052 para US\$ 120.736**, razão de **1,149**. Há deterioração esperada fora da amostra, mas não uma ruptura compatível com overfitting severo. A Regressão Linear também permanece próxima entre treino e teste, porém em patamar inferior de erro; portanto, a vantagem do campeão não decorre apenas de memorizar o treino.

### **Veredito de negócios**

| Decisão | O erro observado é aceitável? | Uso recomendado |
|---|---|---|
| triagem e seleção inicial de comparáveis | **sim, como apoio** | priorizar carteira e sinalizar preços discrepantes |
| definição do preço final de compra ou venda | **não isoladamente** | combinar previsão com comparáveis recentes, vistoria e negociação |
| financiamento, garantia ou laudo | **não como evidência única** | exigir avaliação profissional e política de risco da instituição |
| imóvel de alto padrão | **risco elevado** | revisão humana obrigatória devido à maior dispersão dos erros |

Assim, o resultado é tecnicamente forte para triagem, mas uma margem média de **US\$ 64,7 mil** ainda pode alterar entrada, financiamento e retorno da operação. O número previsto deve iniciar a análise, não encerrá-la.


## **7.2 Resíduos, faixas de preço e interpretação**

Resíduo é a diferença entre valor real e previsto. Resíduos positivos indicam subestimação; negativos indicam superestimação. O padrão desejável é uma nuvem sem estrutura em torno de zero. Um formato de funil sinaliza heterocedasticidade: a amplitude do erro cresce com o preço, situação esperada neste mercado e que justifica avaliar faixas e percentis, não apenas médias.


In [ ]:
def erros_por_faixa(y_true, y_pred) -> pd.DataFrame:
    dados = pd.DataFrame({"real": np.asarray(y_true), "previsto": np.asarray(y_pred)})
    dados["erro_absoluto"] = (dados["real"] - dados["previsto"]).abs()
    dados["erro_percentual"] = dados["erro_absoluto"] / dados["real"]
    dados["faixa_preco"] = pd.qcut(
        dados["real"], 4, labels=["Q1 — menor preço", "Q2", "Q3", "Q4 — maior preço"]
    )
    return (
        dados.groupby("faixa_preco", observed=True)
        .agg(
            registros=("real", "size"),
            preco_mediano=("real", "median"),
            MAE=("erro_absoluto", "mean"),
            MAPE=("erro_percentual", "mean"),
            MdAPE=("erro_percentual", "median"),
            erro_absoluto_p90=("erro_absoluto", lambda s: s.quantile(0.90)),
        )
        .reset_index()
    )

def plot_real_previsto(y_true, y_pred):
    fig, ax = plt.subplots(figsize=(7.5, 6.2))
    ax.scatter(y_true, y_pred, alpha=0.35, s=18, color=CORES["azul"])
    limites = [min(np.min(y_true), np.min(y_pred)), max(np.max(y_true), np.max(y_pred))]
    ax.plot(limites, limites, "--", color=CORES["laranja"], label="previsão perfeita")
    ax.set(title="Preço real × previsto — teste isolado", xlabel="Real (US$)", ylabel="Previsto (US$)")
    ax.legend()
    return _salvar(fig, "07_real_vs_previsto.png")

def plot_residuos(y_true, y_pred):
    residuos = np.asarray(y_true) - np.asarray(y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))
    sns.histplot(residuos, bins=55, ax=axes[0], color=CORES["azul"])
    axes[0].axvline(0, ls="--", color=CORES["laranja"])
    axes[0].set(title="Distribuição dos resíduos", xlabel="Real − previsto (US$)")
    axes[1].scatter(y_pred, residuos, alpha=0.30, s=16, color=CORES["verde"])
    axes[1].axhline(0, ls="--", color=CORES["laranja"])
    axes[1].set(title="Resíduos × previsto", xlabel="Previsto (US$)", ylabel="Resíduo (US$)")
    return _salvar(fig, "08_residuos.png")

campeao = ajustados[campeao_nome]
y_pred = campeao.predict(X_test)
metricas_teste = calcular_metricas(y_test, y_pred)
residuos, erros_abs = y_test.to_numpy() - y_pred, np.abs(y_test.to_numpy() - y_pred)
resumo_residuos = {
    "residuo_medio": float(residuos.mean()), "residuo_mediano": float(np.median(residuos)),
    "erro_absoluto_p90": float(np.quantile(erros_abs, .90)), "erro_absoluto_p95": float(np.quantile(erros_abs, .95)),
    "percentual_erro_ate_50k": float(np.mean(erros_abs <= 50_000)), "percentual_erro_ate_100k": float(np.mean(erros_abs <= 100_000)),
}
faixas = erros_por_faixa(y_test, y_pred)
(ROOT / "outputs/resumo_residuos.json").write_text(json.dumps(resumo_residuos, indent=2, ensure_ascii=False), encoding="utf-8")
faixas.to_csv(ROOT / "outputs/erros_por_faixa_preco.csv", index=False)
display(pd.Series(resumo_residuos, name="resultado").to_frame())
display(faixas.style.format({"preco_mediano": "US$ {:,.0f}", "MAE": "US$ {:,.0f}", "MAPE": "{:.2%}", "MdAPE": "{:.2%}", "erro_absoluto_p90": "US$ {:,.0f}"}))
plot_real_previsto(y_test, y_pred); plt.show()
plot_residuos(y_test, y_pred); plt.show()


O resíduo médio é **+US\$ 2.729** e a mediana é **−US\$ 1.930**; o viés agregado é pequeno, embora isso não elimine erros individuais relevantes. **60,4%** das previsões erram até **US\$ 50 mil** e **82,8%** até **US\$ 100 mil**.

No quartil mais caro, o MAE sobe para aproximadamente **US\$ 131.377** e o percentil 90 do erro absoluto chega a **US\$ 294.696**. O principal insight gerencial é que uma única regra de confiança não serve para toda a carteira: imóveis usuais podem receber triagem mais automatizada; alto padrão deve acionar comparáveis adicionais, vistoria e revisão humana.


## **7.3 Importância por permutação**

Cada variável bruta é embaralhada no teste enquanto as demais permanecem intactas. O aumento do MAE mostra quanto o pipeline completo dependia daquela informação, incluindo as colunas derivadas dela. A técnica mede contribuição preditiva no contexto do modelo; não mede causalidade e pode repartir importância entre variáveis correlacionadas.


In [ ]:
def plot_importancia(importancias: pd.DataFrame):
    dados = importancias.nlargest(12, "importancia").sort_values("importancia")
    fig, ax = plt.subplots(figsize=(8.8, 6))
    ax.barh(dados["variavel"], dados["importancia"], color=CORES["azul"])
    ax.set(title="Importância por permutação — entradas brutas", xlabel="Aumento do MAE ao embaralhar", ylabel="")
    return _salvar(fig, "09_importancia_permutacao.png")

permutacao = permutation_importance(campeao, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=8, random_state=SEED, n_jobs=N_JOBS)
importancias = pd.DataFrame({"variavel": X_test.columns, "importancia": permutacao.importances_mean, "desvio": permutacao.importances_std}).sort_values("importancia", ascending=False, ignore_index=True)
importancias.to_csv(ROOT / "outputs/importancia_permutacao.csv", index=False)
display(importancias.head(12).style.format({"importancia": "{:,.0f}", "desvio": "{:,.0f}"}))
plot_importancia(importancias); plt.show()


A permutação confirma que **localização, padrão construtivo e área** concentram a dependência preditiva. Isso converge com a EDA e com o painel geoespacial: imóveis fisicamente semelhantes podem ter valores diferentes conforme a localização, e localização semelhante não elimina a influência de área e `grade`.

**Insight negocial:** esses fatores devem orientar a seleção de comparáveis e a justificativa de preço, mas não devem ser apresentados como efeitos causais isolados. A importância mostra onde o modelo busca informação, não quanto o preço mudaria se uma variável fosse alterada.


## **7.4 Versionamento e validação do artefato**

Encerrada a avaliação, o pipeline campeão é retreinado na base completa e salvo em `models/v1/modelo_regressao_v1.pkl`. O objeto reúne **engenharia de atributos a partir das colunas brutas, correção da anomalia de quartos aprendida no treino, imputação, escala, encoding, transformação do alvo, correção de retransformação e modelo**. `metricas_v1.json` preserva data UTC, variáveis, validação e métricas do holdout.

A prova final não reutiliza `X` já processado: o arquivo é recarregado e recebe duas linhas diretamente no esquema bruto, sem `price` e sem `id`. Esse teste verifica a melhoria proposta no projeto de referência e demonstra que a previsão de novos imóveis não depende de reconstruir manualmente o preprocessamento.


In [ ]:
from datetime import datetime, timezone
from joblib.externals import cloudpickle

def salvar_versao_modelo(pipeline, metricas: dict, modelo_path, metricas_path) -> None:
    modelo_path.parent.mkdir(parents=True, exist_ok=True)
    # Cloudpickle incorpora as classes autorais do notebook no próprio arquivo.
    # O artefato continua compatível com joblib.load e não depende de módulo auxiliar.
    with modelo_path.open("wb") as arquivo:
        cloudpickle.dump(pipeline, arquivo, protocol=5)
    payload = {"treinado_em_utc": datetime.now(timezone.utc).isoformat(), **metricas}
    metricas_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

pipeline_final = criar_pipeline(preprocessador, modelos[campeao_nome]).fit(X, y)
registro_v1 = {
    "versao": "v1", "modelo": campeao_nome, "alvo": TARGET, "unidade_alvo": "USD", "random_state": SEED,
    "separacao": "GroupShuffleSplit por id, teste de 20%", "registros_treino_validacao": int(len(X_train)),
    "registros_teste": int(len(X_test)), "ids_compartilhados": int(len(set(grupos_train) & set(grupos_test))),
    "criterio_selecao": "menor RMSE médio em GroupKFold de 5 folds no treino",
    "validacao_cruzada": json.loads(cv_resumo.to_json(orient="records")),
    "vitorias_por_fold": json.loads(vitorias_por_fold.to_json(orient="records")),
    "diagnostico_distribuicao_alvo": json.loads(diagnostico_alvo.to_json(orient="records")),
    "metricas_teste": {k: float(v) for k, v in metricas_teste.items()},
    "diagnostico_generalizacao": json.loads(diagnostico.to_json(orient="records")),
    "variaveis_brutas_entrada": list(X.columns), "variaveis_explicativas": MODEL_FEATURES,
    "observacao": "Pipeline bruto→previsão retreinado em toda a base; métricas preservam o holdout independente.",
}
modelo_path, metricas_path = MODEL_DIR / "modelo_regressao_v1.pkl", MODEL_DIR / "metricas_v1.json"
salvar_versao_modelo(pipeline_final, registro_v1, modelo_path, metricas_path)
salvar_csv(pd.concat([X_model, y], axis=1), FINAL_DATA)
ambiente = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "scikit_learn": sklearn.__version__, "matplotlib": matplotlib.__version__, "seaborn": sns.__version__, "scipy": scipy.__version__, "joblib": joblib.__version__, "h3": h3.__version__, "plotly": plotly.__version__}
(ROOT / "environment_info.json").write_text(json.dumps(ambiente, indent=2, ensure_ascii=False), encoding="utf-8")
recarregado = joblib.load(modelo_path)
novos_brutos = pd.read_csv(RAW_DATA).drop(columns=[TARGET, "id"]).head(2)
previsao_teste = recarregado.predict(novos_brutos)
etapas_esperadas = {"engenharia_atributos", "preprocessador", "modelo"}
assert etapas_esperadas.issubset(recarregado.named_steps), "Pipeline v1 incompleto."
assert len(previsao_teste) == len(novos_brutos) and np.isfinite(previsao_teste).all()
artefatos = [modelo_path, metricas_path, PROCESSED_DATA, FINAL_DATA, ROOT / "outputs/relatorio_qualidade_dados.json",
    ROOT / "outputs/diagnostico_distribuicao_alvo.csv", ROOT / "outputs/vitorias_por_fold.csv", MAPS_DIR / "painel_geoespacial.html", MAPS_DIR / "regioes_h3_valor_m2.csv", MAPS_DIR / "resumo_mapa_valor_m2.json"]
display(pd.DataFrame({"artefato": [str(p.relative_to(ROOT)) for p in artefatos], "existe": [p.exists() for p in artefatos]}))
print("Teste bruto→previsão aprovado:", np.round(previsao_teste, 2).tolist())
print("Etapas persistidas:", list(recarregado.named_steps))


O teste de recarga confirma que o arquivo `modelo_regressao_v1.pkl` preserva o pipeline completo: engenharia a partir do esquema bruto, pré-processamento e regressão. Duas linhas originais foram transformadas em previsões finitas sem módulos auxiliares. Assim, a versão v1 não é apenas um estimador serializado; é um artefato **bruto → preço**, acompanhado das métricas, do ambiente e dos dados finais que permitem auditar sua origem.
